# Download B13, crop Việt Nam và lưu NetCDF nén

Notebook tải từng timestamp vào thư mục tạm, giải nén HSD, crop/resample bbox Việt Nam rồi xóa ngay dữ liệu tạm. Kết quả cuối cùng là **một file NetCDF cho một ngày** với cấu trúc `B13(time, latitude, longitude)`.

In [ ]:
%pip install -q satpy pyresample xarray netCDF4 requests

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from datetime import datetime, timedelta
from pathlib import Path
from tempfile import TemporaryDirectory
import bz2
import os
import sys
import warnings

import numpy as np
import requests
import xarray as xr
from pyresample import create_area_def
from satpy import Scene

TARGET_DATE = "20260911"  # UTC, định dạng YYYYMMDD.
INTERVAL_MINUTES = 30
BAND = "B13"
SEGMENTS = ["03", "04", "05"]
VIETNAM_BBOX = (102.0, 8.0, 110.0, 24.5)
GRID_RESOLUTION_DEGREES = 0.02
NOAA_BASE_URL = "https://noaa-himawari9.s3.amazonaws.com"
HTTP_TIMEOUT = (10, 180)
HTTP_RETRIES = 3
USE_GOOGLE_DRIVE = False
DRIVE_OUTPUT_DIR = Path(
    "/content/drive/MyDrive/Rainfall_Nowcasting/Himawari/compact"
)

if "google.colab" in sys.modules and USE_GOOGLE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    OUTPUT_DIR = DRIVE_OUTPUT_DIR
elif "google.colab" in sys.modules:
    OUTPUT_DIR = Path("/content/himawari/compact")
else:
    OUTPUT_DIR = Path("Himawari/data/himawari/compact")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / (
    f"H09_{TARGET_DATE}_{BAND}_Vietnam_{INTERVAL_MINUTES}min.nc"
)
warnings.filterwarnings(
    "ignore",
    message="invalid value encountered in log",
    category=RuntimeWarning,
)

if 24 * 60 % INTERVAL_MINUTES != 0:
    raise ValueError("INTERVAL_MINUTES phải chia hết 1440 phút.")
target_day = datetime.strptime(TARGET_DATE, "%Y%m%d")
slots = [
    target_day + timedelta(minutes=minute)
    for minute in range(0, 24 * 60, INTERVAL_MINUTES)
]
vietnam_area = create_area_def(
    "vietnam_latlon",
    {"proj": "longlat", "datum": "WGS84"},
    area_extent=VIETNAM_BBOX,
    resolution=(GRID_RESOLUTION_DEGREES, GRID_RESOLUTION_DEGREES),
    units="degrees",
)
print(f"Sẽ xử lý {len(slots)} timestamp, {INTERVAL_MINUTES} phút/lần.")
print("Output:", OUTPUT_FILE.resolve())

In [ ]:
def make_filename(slot: datetime, segment: str) -> str:
    return (
        f"HS_H09_{slot:%Y%m%d}_{slot:%H%M}_{BAND}_"
        f"FLDK_R20_S{segment}10.DAT"
    )


def make_url(slot: datetime, filename: str) -> str:
    key = f"AHI-L1b-FLDK/{slot:%Y/%m/%d/%H%M}/{filename}.bz2"
    return f"{NOAA_BASE_URL}/{key}"


def download_file(url: str, destination: Path) -> None:
    temporary = destination.with_suffix(destination.suffix + ".part")
    for attempt in range(1, HTTP_RETRIES + 1):
        try:
            with requests.get(url, stream=True, timeout=HTTP_TIMEOUT) as response:
                response.raise_for_status()
                with temporary.open("wb") as output_file:
                    for chunk in response.iter_content(1024 * 1024):
                        if chunk:
                            output_file.write(chunk)
            os.replace(temporary, destination)
            return
        except requests.RequestException:
            temporary.unlink(missing_ok=True)
            if attempt == HTTP_RETRIES:
                raise


def decompress_file(source: Path, destination: Path) -> None:
    with bz2.open(source, "rb") as input_file:
        with destination.open("wb") as output_file:
            while chunk := input_file.read(1024 * 1024):
                output_file.write(chunk)


def prepare_segment(slot: datetime, segment: str, work_dir: Path) -> Path:
    filename = make_filename(slot, segment)
    compressed_path = work_dir / f"{filename}.bz2"
    dat_path = work_dir / filename
    download_file(make_url(slot, filename), compressed_path)
    decompress_file(compressed_path, dat_path)
    return dat_path

In [ ]:
def process_slot(slot: datetime) -> xr.DataArray:
    with TemporaryDirectory(prefix="himawari_") as temporary_dir:
        work_dir = Path(temporary_dir)
        with ThreadPoolExecutor(max_workers=len(SEGMENTS)) as executor:
            dat_files = list(
                executor.map(
                    lambda segment: prepare_segment(slot, segment, work_dir),
                    SEGMENTS,
                )
            )
        scene = Scene(reader="ahi_hsd", filenames=[str(path) for path in dat_files])
        scene.load([BAND], calibration="brightness_temperature")
        cropped_scene = scene.crop(ll_bbox=VIETNAM_BBOX)
        resample_scene = Scene()
        resample_scene[BAND] = cropped_scene[BAND]
        latlon_scene = resample_scene.resample(
            vietnam_area,
            datasets=[BAND],
            resampler="nearest",
            unload=False,
        )
        data = latlon_scene[BAND].rename({"y": "latitude", "x": "longitude"})
        data = data.astype(np.float32).compute().sortby("latitude")

    timestamp = np.datetime64(slot.strftime("%Y-%m-%dT%H:%M"))
    data = data.expand_dims(time=[timestamp]).copy(deep=True)
    data.name = BAND
    data.attrs = {
        "long_name": "Himawari-9 AHI B13 brightness temperature",
        "units": "K",
        "source": "NOAA Open Data, AHI Level-1b HSD",
        "spatial_resolution_degrees": GRID_RESOLUTION_DEGREES,
    }
    data.latitude.attrs = {"standard_name": "latitude", "units": "degrees_north"}
    data.longitude.attrs = {"standard_name": "longitude", "units": "degrees_east"}
    return data

In [ ]:
frames = []
failures = []
for index, slot in enumerate(slots, start=1):
    try:
        frames.append(process_slot(slot))
        print(f"[{index:02d}/{len(slots)}] OK {slot:%Y-%m-%d %H:%M} UTC")
    except Exception as error:
        failures.append((slot, str(error)))
        print(f"[{index:02d}/{len(slots)}] LỖI {slot:%Y-%m-%d %H:%M}: {error}")

if not frames:
    raise RuntimeError("Không xử lý được timestamp nào.")
daily_data = xr.concat(frames, dim="time").sortby("time")
dataset = daily_data.to_dataset(name=BAND)
dataset = dataset.drop_vars("crs", errors="ignore")
dataset.time.attrs = {"standard_name": "time", "timezone": "UTC"}
dataset.attrs = {
    "title": "Himawari-9 B13 cropped to Vietnam bounding box",
    "bbox": "102E, 8N, 110E, 24.5N",
    "source_format": "Himawari Standard Data (HSD)",
}
chunk_shape = (
    1,
    min(256, dataset.sizes["latitude"]),
    min(256, dataset.sizes["longitude"]),
)
encoding = {
    BAND: {
        "dtype": "float32",
        "zlib": True,
        "complevel": 4,
        "shuffle": True,
        "chunksizes": chunk_shape,
    }
}
temporary_output = OUTPUT_FILE.with_suffix(".nc.part")
dataset.to_netcdf(
    temporary_output,
    engine="netcdf4",
    format="NETCDF4",
    encoding=encoding,
)
os.replace(temporary_output, OUTPUT_FILE)
size_mb = OUTPUT_FILE.stat().st_size / 1024**2
print(f"\nĐã lưu {len(frames)}/{len(slots)} timestamp: {OUTPUT_FILE}")
print(f"Dung lượng: {size_mb:.2f} MB")
print("Kích thước:", dict(dataset.sizes))
if failures:
    print("Timestamp lỗi:")
    for slot, error in failures:
        print(f"- {slot:%Y-%m-%d %H:%M} UTC: {error}")

In [ ]:
with xr.open_dataset(OUTPUT_FILE) as saved_dataset:
    print(saved_dataset)
    print("time đầu:", saved_dataset.time.values[0])
    print("time cuối:", saved_dataset.time.values[-1])
    print("missing ratio:", float(saved_dataset[BAND].isnull().mean()))